# Inferring pooled GRNs from homogeneous cell populations with Perturb-seq-based validation

scCAFM predicts a cell-specific GRN for each individual cell. In a homogeneous population such as a cell line, averaging regulatory edge scores across cells produces a pooled GRN that emphasizes interactions consistently predicted across the population. Here, we infer this network using only non-targeting control cells from a K562 Perturb-seq dataset.

We then use the perturbation measurements as an independent source of validation. For each highly ranked TF-to-target prediction, we compare target-gene expression between held-out non-targeting cells and cells in which the source TF was perturbed. The Wasserstein distance summarizes how strongly the two expression distributions differ: a larger value indicates a larger perturbation-associated shift. Perturbed cells are not used to infer the pooled GRN or select the evaluated edges.

A perturbation-associated expression shift supports the biological relevance of a predicted edge, but it does not establish a direct regulatory interaction. Responses may be indirect or off-target, and the Wasserstein distance measures the magnitude of distributional change rather than whether expression increases or decreases.

## 1. Set up the tutorial

The model files are read from `assets/`. The K562 file under `tutorial_data/perturbseq_edge_validation/` contains raw counts after perturbation-level QC, cell-level perturbation-effect QC, and basic cell/gene filtering. All preprocessing needed for scCAFM is performed explicitly below and does not modify the stored file.

In [1]:
from pathlib import Path
import warnings

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split

warnings.filterwarnings(
    "ignore",
    message="Mismatch dtype between input and weight",
)

from sccafm import (
    GRNInferencer,
    ScPreprocessor,
    evaluate_perturbseq_grn,
    load_vocab_json,
    resolve_model_assets,
)


REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

MODEL_SOURCE = REPO_ROOT / "assets"
DATA_PATH = (
    REPO_ROOT
    / "tutorial_data"
    / "perturbseq_edge_validation"
    / "K562.h5ad"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the raw-count K562 dataset at: {DATA_PATH}"
    )

assets = resolve_model_assets(MODEL_SOURCE)
token_dict = load_vocab_json(assets.vocab)
human_tfs = pd.read_csv(assets.human_tfs)
print("Tutorial files are ready.")

Tutorial files are ready.


## 2. Examine the K562 data

K562 is a human myeloid leukaemia cell line with many measured gene perturbations. The retained cells passed the established perturbation-strength and successful-knockdown filters. The tutorial file keeps the broad raw-count gene matrix rather than restricting targets to perturbed genes, so scCAFM can select an informative model input while held-out raw counts remain available for validation.

### Prepare your own Perturb-seq data

The K562 file shows the metadata needed to infer a pooled control-cell GRN and validate its edges with perturbations:

- Store raw, finite and non-negative counts in `adata.X`. The tutorial applies its own normalization and `log1p` transformation, so do not preprocess the matrix a second time.
- Put unique gene symbols in `adata.var["gene_name"]`. The matrix column identifiers in `adata.var_names` may remain Ensembl IDs or other stable feature IDs.
- Add `adata.obs["gene"]` with the perturbed gene symbol for each perturbed cell and `"non-targeting"` for control cells. These symbols must match `adata.var["gene_name"]` and the human TF catalogue.
- Add `adata.obs["gene_id"]` for the perturbation identifier, plus `adata.obs["species"] = "human"` and an appropriate `adata.obs["disease"]` value. The demo uses `"chronic myeloid leukemia"`.
- Use unique cell identifiers. The tutorial requires more than 100 cells for a TF perturbation to be eligible; adjust that filtering rule if a smaller experiment has adequate replication.
- `adata.uns["basic_filter"]` and `adata.uns["perturbation_qc"]` are optional provenance summaries. They improve the overview table but are not used to calculate Wasserstein distances.

A minimal in-memory setup looks like:

```python
adata = sc.read_h5ad("my_perturbseq_counts.h5ad")
adata.obs["gene"] = perturbation_symbols
adata.obs["gene_id"] = perturbation_ids
adata.obs["species"] = "human"
adata.obs["disease"] = "my condition"
adata.var["gene_name"] = measured_gene_symbols
```

If your perturbation column or control label has another name, update the preparation cells and pass the same values as `perturbation_key` and `control_label` to `evaluate_perturbseq_grn()`. Perturbation labels are used only for validation; non-targeting cells remain the only cells used to infer the pooled GRN.


In [2]:
raw = sc.read_h5ad(DATA_PATH, backed="r")

required_obs = {"gene", "gene_id", "species", "disease"}
required_var = {"gene_name"}
missing_obs = required_obs.difference(raw.obs.columns)
missing_var = required_var.difference(raw.var.columns)
if missing_obs or missing_var:
    raise KeyError(
        "The K562 file is missing required metadata: "
        + ", ".join(sorted(missing_obs | missing_var))
    )

perturbation_counts = raw.obs["gene"].astype(str).value_counts()
qc = dict(raw.uns.get("perturbation_qc", {}))
basic_filter = dict(raw.uns.get("basic_filter", {}))
dataset_overview = pd.DataFrame(
    [
        {
            "Dataset": "K562",
            "Species": raw.obs["species"].astype(str).iloc[0],
            "Expression scale": "raw counts",
            "Original cells": basic_filter.get(
                "original_n_obs", raw.n_obs
            ),
            "After perturbation-level QC": qc.get(
                "cells_after_perturbation_level_qc", raw.n_obs
            ),
            "Retained cells": raw.n_obs,
            "Retained genes": raw.n_vars,
            "Non-targeting controls": perturbation_counts.get(
                "non-targeting", 0
            ),
            "Retained perturbations": (
                raw.obs["gene"].astype(str).nunique() - 1
            ),
        }
    ]
)
raw.file.close()

numeric_columns = [
    "Original cells",
    "After perturbation-level QC",
    "Retained cells",
    "Retained genes",
    "Non-targeting controls",
    "Retained perturbations",
]
display(
    dataset_overview.style.format(
        {column: "{:,.0f}" for column in numeric_columns}
    )
)

,Dataset,Species,Expression scale,Original cells,After perturbation-level QC,Retained cells,Retained genes,Non-targeting controls,Retained perturbations
0,K562,human,raw counts,"310,385","192,648","162,751","8,563","10,691","1,092"


## 3. Prepare the expression data

The stored cells have already passed perturbation QC. We now split non-targeting cells reproducibly: 80% are used to infer the pooled GRN and 20% are held out for validation. Retained TF perturbations with more than 100 cells are eligible for validation.

`ScPreprocessor` performs model preprocessing only on the inference controls. It normalizes each cell to a total of `10000`, applies `log1p`, retains all available catalogue TFs, and selects 1,000 additional highly variable genes. The held-out cells are used only for validation.


In [3]:
raw = sc.read_h5ad(DATA_PATH, backed="r")
obs = raw.obs.copy()
measured_symbols = set(
    raw.var["gene_name"].astype(str).str.upper()
)
vocabulary_symbols = set(
    token_dict["gene_symbol"].dropna().astype(str).str.upper()
)
catalog_tfs = set(
    human_tfs["TF"].dropna().astype(str).str.upper()
)

perturbation_counts = obs["gene"].astype(str).value_counts()
eligible_perturbed_tfs = sorted(
    gene
    for gene, count in perturbation_counts.items()
    if gene != "non-targeting"
    and count > 100
    and gene.upper() in measured_symbols
    and gene.upper() in vocabulary_symbols
    and gene.upper() in catalog_tfs
)
available_catalog_tfs = sorted(
    measured_symbols & vocabulary_symbols & catalog_tfs
)

control_indices = np.flatnonzero(
    obs["gene"].astype(str).to_numpy() == "non-targeting"
)
inference_indices, validation_control_indices = train_test_split(
    control_indices,
    test_size=0.2,
    random_state=0,
)
perturbed_indices = np.flatnonzero(
    obs["gene"].astype(str).isin(eligible_perturbed_tfs).to_numpy()
)
validation_indices = np.concatenate(
    [np.sort(validation_control_indices), np.sort(perturbed_indices)]
)

inference_raw = raw[np.sort(inference_indices), :].to_memory()
validation_adata = raw[validation_indices, :].to_memory()
raw.file.close()

preprocessor = ScPreprocessor(
    min_genes=200,
    min_cells=3,
    max_pct_counts_mt=20.0,
    target_sum=10000,
    log1p=True,
    n_top_genes=1000,
    hvg_flavor="seurat",
    subset_hvg=True,
    remove_mito_genes=True,
    remove_ribo_genes=True,
    remove_hb_genes=True,
    token_dict=token_dict,
    gene_key="gene_name",
    preserve_gene_names=available_catalog_tfs,
    hvg_exclude_gene_names=available_catalog_tfs,
    hvg_exclude_preserved_genes=True,
    sanitize_X=True,
    inplace=False,
)
inference_adata = preprocessor(inference_raw)
# Keep the exact raw-count columns used by the model. This also removes
# ambiguous duplicate symbols from the broader source matrix.
validation_adata = validation_adata[
    :, inference_adata.var_names
].copy()
sc.pp.log1p(validation_adata)

prepared_overview = pd.DataFrame(
    [
        {
            "Inference controls": inference_adata.n_obs,
            "Validation controls": len(validation_control_indices),
            "Validation perturbed cells": len(perturbed_indices),
            "Eligible perturbed TFs": len(eligible_perturbed_tfs),
            "Retained catalogue TFs": int(
                inference_adata.var.get(
                    "preserved_gene",
                    pd.Series(False, index=inference_adata.var_names),
                ).sum()
            ),
            "Additional HVGs": int(
                inference_adata.var.get(
                    "hvg_target_gene",
                    pd.Series(False, index=inference_adata.var_names),
                ).sum()
            ),
            "Model genes": inference_adata.n_vars,
        }
    ]
)
display(
    prepared_overview.style.format(
        {column: "{:,.0f}" for column in prepared_overview.columns}
    )
)

,Inference controls,Validation controls,Validation perturbed cells,Eligible perturbed TFs,Retained catalogue TFs,Additional HVGs,Model genes
0,"8,537","2,139","16,648",94,924,"1,000","1,924"


## 4. Infer a pooled GRN

The inference cells represent one non-targeting K562 population, so we average their cell-specific networks to obtain one pooled GRN. The model uses FA2 on one GPU. The returned network is unfiltered because edge selection is performed only during validation.

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires one CUDA GPU.")

inferencer = GRNInferencer.from_pretrained(
    MODEL_SOURCE,
    device="cuda:0",
    attention_backend="fa2",
    max_length=2048,
    species_key="species",
    disease_key="disease",
)

pooled_grn = inferencer.infer_pooled(
    inference_adata,
    batch_size=8,
    gene_key="gene_name",
    score_threshold=None,
    top_k_edges=None,
)

pooled_overview = pd.DataFrame(
    [
        {
            "Inference cells": pooled_grn.n_cells,
            "Source TFs": len(pooled_grn.source_genes),
            "Target genes": len(pooled_grn.target_genes),
            "Candidate matrix": (
                f"{pooled_grn.shape[0]:,} × {pooled_grn.shape[1]:,}"
            ),
        }
    ]
)
display(pooled_overview)

,Inference cells,Source TFs,Target genes,Candidate matrix
0,8537,924,1924,"924 × 1,924"


## 5. Validate the top regulatory edges

We remove self-edges and select the 100 highest-scoring TF-to-target predictions. Perturbed cells are not used to infer or choose these edges.

For each selected edge, `evaluate_perturbseq_grn()` compares target-gene expression between held-out non-targeting cells and cells where the source TF was perturbed. A larger Wasserstein distance indicates a larger shift between the two expression distributions.


In [5]:
TOP_K_EDGES = 100

evaluation = evaluate_perturbseq_grn(
    pooled_grn,
    validation_adata,
    perturbation_key="gene",
    control_label="non-targeting",
    top_k_edges=TOP_K_EDGES,
    gene_key="gene_name",
)
validated_edges = (
    evaluation.to_edge_table()
    .reset_index(drop=True)
)

validation_summary = pd.DataFrame(
    [
        {
            "Candidate edges": evaluation.n_candidates,
            "Evaluated edges": evaluation.n_evaluated_edges,
            "Perturbed TFs": evaluation.n_perturbed_tfs,
            "Validation controls": evaluation.n_control_cells,
            "Validation perturbed cells": evaluation.n_perturbed_cells,
            "Mean Wasserstein distance": (
                evaluation.mean_wasserstein_distance
            ),
        }
    ]
)
display(
    validation_summary.style.format(
        {
            "Candidate edges": "{:,.0f}",
            "Evaluated edges": "{:,.0f}",
            "Perturbed TFs": "{:,.0f}",
            "Validation controls": "{:,.0f}",
            "Validation perturbed cells": "{:,.0f}",
            "Mean Wasserstein distance": "{:.4f}",
        }
    )
)

display(
    validated_edges.head(10).style.format(
        {
            "rank": "{:,.0f}",
            "score": "{:.4f}",
            "n_control": "{:,.0f}",
            "n_perturbed": "{:,.0f}",
            "wasserstein_distance": "{:.4f}",
        }
    )
)

,Candidate edges,Evaluated edges,Perturbed TFs,Validation controls,Validation perturbed cells,Mean Wasserstein distance
0,"180,762",100,94,"2,139","16,648",0.4260


,rank,Gene1,Gene2,score,n_control,n_perturbed,wasserstein_distance
0,1,SNRPB,NPM1,0.1626,"2,139",112,0.9075
1,2,SNRPB,ENO1,0.1625,"2,139",112,0.7578
2,3,SNRPD1,NPM1,0.1625,"2,139",112,0.8468
3,4,SNRPD1,ENO1,0.1624,"2,139",112,0.8151
4,5,SNRPB,YBX1,0.1623,"2,139",112,0.6897
5,6,SNRPD1,YBX1,0.1622,"2,139",112,0.5721
6,7,SNRPB,NME2,0.1621,"2,139",112,0.6295
7,8,SNRPD1,NME2,0.1620,"2,139",112,0.5682
8,9,SFPQ,NPM1,0.1618,"2,139",209,0.9075
9,10,SFPQ,ENO1,0.1617,"2,139",209,1.2784


## 6. Optional: save the validation table

The cell below is disabled by default. When enabled, it saves all 100 evaluated edges with these columns:

```text
rank,Gene1,Gene2,score,n_control,n_perturbed,wasserstein_distance
```

You may change `TOP_K_EDGES` before evaluation to examine a larger or smaller ranked set. A larger set includes weaker model predictions and takes longer to validate.

In [6]:
SAVE_VALIDATED_EDGES = False

if SAVE_VALIDATED_EDGES:
    output_dir = REPO_ROOT / "results"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"K562_top{TOP_K_EDGES}_validated_edges.csv"
    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite the existing file: {output_path}"
        )
    validated_edges.to_csv(output_path, index=False)
    print(f"Saved validated edges to {output_path}")
else:
    print(
        "CSV export is disabled. Set SAVE_VALIDATED_EDGES = True "
        "to save the table."
    )

CSV export is disabled. Set SAVE_VALIDATED_EDGES = True to save the table.


## What you learned

`pooled_grn` is inferred from preprocessed non-targeting K562 controls. `validated_edges` contains its top 100 non-self predictions together with their Wasserstein distances.

The scCAFM score ranks predicted regulatory edges, while the Wasserstein distance summarizes the observed expression shift after perturbing the source TF. A large distance supports a perturbation response but does not prove a direct interaction.
